<img src="https://sfile.chatglm.cn/images-ppt/cfe256b11254.jpg" width="500" align="center">

# От блокнота к сервису — обучаем модель и упаковываем в API

**Роль:** Преподаватель по ML
**Уровень:** средний (основы Python + базовое понимание ML)
**Время прохождения:** ~90–120 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- обучите реальную модель классификации изображений (кошки vs собаки);
- **своими руками** сохраните модель и загрузите обратно;
- поймёте, чем «обучение в блокноте» отличается от «модель в продакшене»;
- **сами** напишете FastAPI-сервис, принимающий картинку и возвращающий прогноз;
- протестируете сервис прямо из блокнота;
- увидите типичные ошибки и научитесь их читать.

### Принцип этого блокнота

> Вы — автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких «чёрных ящиков».

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | Знакомство с данными | Скачиваем датасет, смотрим картинки, считаем баланс классов |
| 3 | Визуальный анализ данных | Распределение размеров, цветов, интерактивный браузер |
| 4 | Обучаем модель | DataLoaders, learner, fine_tune — с подробным разбором |
| 5 | Оцениваем модель | Loss-кривые, confusion matrix, топ ошибок |
| 6 | Разбираем predict() | Пошагово: препроцессинг → прямой проход → softmax → порог |
| 7 | Интерактивный тестер | Слайдер по датасету + порог уверенности |
| 8 | Сохраняем и загружаем | learn.export() vs torch.save() — сравнение |
| 9 | Блокнот vs продакшен | Визуальная диаграмма различий |
| 10 | Пишем FastAPI-сервис | model_service.py + main.py — построчно |
| 11 | Запускаем и тестируем | 6 тестов, включая «плохие» запросы |
| 12 | Типичные ошибки | Справочник с решениями |
| 13 | Практические задания | 5 заданий для самостоятельной работы |

---

---
## Шаг 1. Подготовка окружения

Нам понадобятся:

| Библиотека | Зачем |
|-----------|-------|
| **fastai** | Обучение модели классификации изображений (high-level API над PyTorch) |
| **fastapi** | Создание HTTP-API для модели |
| **uvicorn** | ASGI-сервер для запуска FastAPI |
| **python-multipart** | Приём файлов через HTTP |
| **httpx** | HTTP-клиент для тестирования нашего API |
| **ipywidgets** | Интерактивные виджеты (слайдеры, аккордеоны) |

> Каждая библиотека установится с зависимостями. В Colab fastai уже предустановлен, но мы перестрахуемся.


In [ ]:
# ============================================================
# ШАГ 1: Установка зависимостей
# ============================================================
# Флаг -q = quiet (тихий режим, меньше вывода)
# В Colab fastai уже есть, но обновление не повредит

!pip install -q fastai fastapi uvicorn python-multipart Pillow httpx ipywidgets

# Включаем поддержку виджетов в Colab
# 2>/dev/null подавляет предупреждения, || true гарантирует что ячейка не упадёт
!jupyter nbextension enable --py widgetsnbextension 2>/dev/null || true

# ============================================================
# Проверяем версии — это важно для воспроизводимости!
# Если у вас другие версии, некоторые функции могут работать иначе
# ============================================================
import sys
print(f'Python:  {sys.version}')

import torch
print(f'PyTorch: {torch.__version__}')

import fastai
print(f'fastai:  {fastai.__version__}')

import fastapi
print(f'fastapi: {fastapi.__version__}')

import ipywidgets as widgets
print(f'ipywidgets: OK')

# ============================================================
# Определяем устройство: GPU или CPU
# GPU (cuda) ускоряет обучение в 10-50 раз!
# В Colab: Runtime -> Change runtime type -> GPU
# ============================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nУстройство: {device}')
if device == 'cpu':
    print('⚠️  CPU — обучение будет идти медленнее. Включите GPU в настройках Colab!')
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✅ GPU: {gpu_name} — обучение будет быстрым!')

---
## Шаг 2. Знакомство с данными

Прежде чем обучать модель, нужно **понять данные**. Это правило номер один в ML: сначала данные, потом модель.

Мы работаем с **Oxford-IIIT Pet Dataset** — 7349 фотографий кошек и собак 37 пород.

<img src="https://sfile.chatglm.cn/images-ppt/1eabcf97adfd.png" width="600" align="center">

*Сверху: примеры изображений из датасета. Обратите внимание на разный размер, фон, освещение.*

### Как отличить кошку от собаки в этом датасете?

Секрет в **имени файла**:
- Кошки: первая буква **ЗАГЛАВНАЯ** — `Abyssinian_1.jpg`, `Persian_23.jpg`
- Собаки: первая буква **строчная** — `beagle_5.jpg`, `pug_100.jpg`

Это неочевидное правило — именно поэтому важно смотреть на данные руками!


In [ ]:
# ============================================================
# ШАГ 2: Загрузка датасета
# ============================================================
# from fastai.vision.all import * импортирует ВСЁ нужное для vision-задач
# Это включает: DataLoader, Learner, метрики, трансформации и т.д.
from fastai.vision.all import *

# untar_data() делает следующее:
# 1. Скачивает .tar-архив с URL (если ещё не скачан)
# 2. Распаковывает в ~/.fastai/data/
# 3. Возвращает путь к распакованной папке
# URLs.PETS — константа с URL датасета Oxford-IIIT Pet
path = untar_data(URLs.PETS)

print(f'Датасет загружен в: {path}')
print(f'Папки внутри: {path.ls()}')
# Ожидаем: ['images', 'annotations'] — картинки и разметка

# ============================================================
# Находим все картинки
# ============================================================
# get_image_files() рекурсивно обходит папку и находит все файлы-картинки
img_path = path / 'images'           # подпапка с изображениями
image_files = get_image_files(img_path)  # список путей к файлам

print(f'\nВсего изображений: {len(image_files)}')
print(f'Первые 5 файлов:')
for f in image_files[:5]:
    print(f'  {f.name}')

In [ ]:
# ============================================================
# Разбираемся с метками (labels)
# ============================================================
# Правило датасета: кошки начинаются с заглавной буквы, собаки — со строчной
# Это СТАНДАРТНОЕ правило именно для Oxford PETS

# Демонстрируем правило на примерах:
print('Примеры файлов и их метки:')
print('-' * 50)
for f in sorted(image_files)[:10]:
    # f.name[0] — первая буква имени файла
    # isupper() — заглавная ли она?
    animal = '🐱 КОШКА' if f.name[0].isupper() else '🐶 СОБАКА'
    print(f'  {f.name:30s} -> {animal}')

print(f'\n...и так далее, всего {len(image_files)} файлов')

# ============================================================
# Создаём функцию-метку
# ============================================================
# Эта функция будет использоваться DataLoaders для разметки данных
# Вход: имя файла (строка)
# Выход: True (кошка) или False (собака)
def is_cat(filename):
    """Определяет, кошка ли на картинке, по имени файла.
    
    Oxford PETS dataset naming convention:
    - Кошки: первая буква ЗАГЛАВНАЯ (Abyssinian, Persian, ...)
    - Собаки: первая буква строчная (beagle, pug, ...)
    
    Args:
        filename: имя файла (строка), например 'Abyssinian_1.jpg'
    
    Returns:
        True если кошка, False если собака
    """
    return filename[0].isupper()

# ============================================================
# Считаем баланс классов
# ============================================================
# Сбалансированные классы — хорошо, несбалансированные — нужно учитывать
cats = sum(1 for f in image_files if is_cat(f.name))
dogs = len(image_files) - cats

print(f'\nБаланс классов:')
print(f'  Кошки: {cats:5d} ({cats/len(image_files):.1%})')
print(f'  Собаки: {dogs:5d} ({dogs/len(image_files):.1%})')
print(f'\nКлассы {"✅ сбалансированы" if abs(cats-dogs) < 500 else "⚠️ несбалансированы"} — разница {abs(cats-dogs)} изображений')

---
## Шаг 3. Визуальный анализ данных

Прежде чем обучать — **посмотрите на данные!** Это самый важный шаг, который многие пропускают.

Мы визуализируем:
1. **Распределение классов** — сколько кошек vs собак (пирог)
2. **Размеры изображений** — насколько они разные (гистограмма)
3. **Примеры картинок** — интерактивный браузер со слайдером
4. **Средние цвета** — какие цвета преобладают (RGB-визуализация)


In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ 1: Распределение классов
# ============================================================
# Диаграмма-пирог показывает баланс классов
# Если один класс сильно преобладает — модель может «лениться»
# и всегда предсказывать мажоритарный класс

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Пирог ---
sizes = [cats, dogs]
labels = [f'Кошки\n({cats})', f'Собаки\n({dogs})']
colors_pie = ['#FF6B6B', '#4ECDC4']
explode = (0.05, 0.05)  # чуть выносим куски для красоты

axes[0].pie(sizes, labels=labels, colors=colors_pie, explode=explode,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Баланс классов', fontsize=14, fontweight='bold')

# --- Барчарт по породам ---
# Извлекаем породу из имени файла (всё до последнего '_')
from collections import Counter
breeds = []
for f in image_files:
    # 'Abyssinian_1.jpg' -> 'Abyssinian'
    breed = f.name.rsplit('_', 1)[0]
    breeds.append(breed)

breed_counts = Counter(breeds)
top_breeds = breed_counts.most_common(15)  # топ-15 пород

breed_names = [b[0] for b in top_breeds]
breed_values = [b[1] for b in top_breeds]
bar_colors = ['#FF6B6B' if n[0].isupper() else '#4ECDC4' for n in breed_names]

axes[1].barh(breed_names, breed_values, color=bar_colors)
axes[1].set_xlabel('Количество изображений')
axes[1].set_title('Топ-15 пород', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()  # самая частая порода сверху

plt.tight_layout()
plt.show()

print('\nКрасные = кошки, зелёные = собаки')
print(f'Всего пород: {len(breed_counts)}')
print(f'Самая частая порода: {breed_counts.most_common(1)[0]}')
print(f'Самая редкая порода: {breed_counts.most_common()[-1]}')

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ 2: Размеры изображений
# ============================================================
# Нейросети требуют фиксированный размер входа (224x224 для ResNet)
# Но исходные картинки разного размера — их нужно масштабировать
# Понимание распределения размеров помогает выбрать стратегию ресайза

from PIL import Image
import numpy as np

# Собираем размеры (выборка, т.к. полный обход долгий)
sample_size = min(500, len(image_files))
sample_indices = np.random.choice(len(image_files), sample_size, replace=False)

widths, heights = [], []
for idx in sample_indices:
    img = Image.open(image_files[idx])
    w, h = img.size
    widths.append(w)
    heights.append(h)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Ширина ---
axes[0].hist(widths, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(x=224, color='red', linestyle='--', linewidth=2, label='224 (целевой)')
axes[0].set_xlabel('Ширина (пиксели)')
axes[0].set_ylabel('Количество')
axes[0].set_title('Распределение ширин')
axes[0].legend()

# --- Высота ---
axes[1].hist(heights, bins=30, color='coral', alpha=0.7, edgecolor='white')
axes[1].axvline(x=224, color='red', linestyle='--', linewidth=2, label='224 (целевой)')
axes[1].set_xlabel('Высота (пиксели)')
axes[1].set_ylabel('Количество')
axes[1].set_title('Распределение высот')
axes[1].legend()

# --- Scatter: ширина vs высота ---
axes[2].scatter(widths, heights, alpha=0.3, s=10, c='purple')
axes[2].axhline(y=224, color='red', linestyle='--', alpha=0.5)
axes[2].axvline(x=224, color='red', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Ширина')
axes[2].set_ylabel('Высота')
axes[2].set_title('Ширина vs Высота')
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

print(f'Медианный размер: {int(np.median(widths))}x{int(np.median(heights))}')
print(f'Минимальный: {min(widths)}x{min(heights)}')
print(f'Максимальный: {max(widths)}x{max(heights)}')
print('\nКрасная пунктирная линия = 224 пикселя (целевой размер для модели)')
print('Большинство картинок БОЛЬШЕ 224x224 — модель будет их уменьшать (downscale)')

In [ ]:
# ============================================================
# ИНТЕРАКТИВ: Браузер датасета
# ============================================================
# Перемещайте слайдер, чтобы листать картинки
# Для каждой картинки показываем: имя файла, метку, размер, породу

import ipywidgets as widgets
from IPython.display import display, clear_output

idx_slider = widgets.IntSlider(
    value=0,                     # начальное значение
    min=0,                       # минимум
    max=len(image_files)-1,      # максимум
    step=1,                      # шаг
    description='Индекс:',       # подпись
    layout=widgets.Layout(width='500px')  # ширина слайдера
)

output = widgets.Output()  # область вывода — будет перерисовываться

def show_image(idx):
    """Показывает картинку по индексу с подробной информацией.
    
    Args:
        idx: индекс в списке image_files
    """
    with output:
        clear_output(wait=True)  # очищаем предыдущий вывод
        
        # Открываем картинку
        f = image_files[idx]
        img = PILImage.create(f)
        
        # Определяем метку
        label = '🐱 КОШКА' if is_cat(f.name) else '🐶 СОБАКА'
        breed = f.name.rsplit('_', 1)[0]  # порода = всё до последнего '_'
        
        # Рисуем
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'{label} — {breed}', fontsize=14, fontweight='bold')
        plt.show()
        
        # Текстовая информация
        print(f'Файл:  {f.name}')
        print(f'Размер: {img.size[0]}x{img.size[1]} пикселей')
        print(f'Метка:  {label}')
        print(f'Порода: {breed}')

# Связываем слайдер с функцией
widgets.interact(show_image, idx=idx_slider)
display(output)

### Что вы заметили?

Пролистайте 20–30 картинок и обратите внимание:
- **Разные размеры** — модель требует одинаковый размер входа (224x224)
- **Разный фон** — диван, улица, трава... модель должна быть устойчива к фону
- **Разная поза** — морда анфас, профиль, со спины... модель должна узнавать в любой позе
- **Разное освещение** — тёмные, светлые, жёлтые... модель не должна зависеть от цвета света
- **Некоторые фото обрезаны** — только часть животного. Это сложно даже для человека!

> Все эти проблемы модель решает **сама** в процессе обучения — если данных достаточно.

---

---
## Шаг 4. Обучаем модель

<img src="https://sfile.chatglm.cn/images-ppt/1eabcf97adfd.png" width="600" align="center">

*Transfer learning: мы берём модель, уже обученную на ImageNet (миллионы картинок), и «дообучаем» на нашу задачу.*

### Что такое transfer learning?

Представьте, что вы учитесь рисовать портреты:
1. Сначала вы научились **видеть** (линии, формы, цвета) — это **предобучение на ImageNet**
2. Теперь вы учитесь **различать кошек и собак** — это **fine-tuning на нашем датасете**

Мы не учим модель «с нуля» — она уже умеет видеть! Мы только учим её нашему конкретному различию.

### Три ключевых объекта fastai

| Объект | Что делает | Аналогия |
|--------|-----------|----------|
| `DataLoaders` | Подготавливает данные (батчи, трансформации) | Конвейер на заводе |
| `Learner` | Связывает данные + модель + оптимизатор | Ученик с учебником |
| `fine_tune()` | Запускает обучение по стратегии transfer learning | Процесс обучения |


In [ ]:
# ============================================================
# ШАГ 4a: Создаём DataLoaders — конвейер данных
# ============================================================
# DataLoaders — это «трубопровод», который:
# 1. Берёт сырые файлы с диска
# 2. Разделяет на train/valid (80%/20%)
# 3. Применяет метки (кошка/собака)
# 4. Трансформирует (ресайзит до 224x224)
# 5. Группирует в батчи по 32 штуки
# 6. Отдаёт модели по запросу

dls = ImageDataLoaders.from_name_func(
    path,                              # корневая папка (для кэша)
    get_image_files(img_path),         # список файлов — какие картинки использовать
    valid_pct=0.2,                     # 20% данных -> валидация (модель их НЕ видит при обучении!)
    seed=42,                           # фиксируем random seed (воспроизводимость результатов)
    label_func=is_cat,                 # функция метки: True=кошка, False=собака
    item_tfms=Resize(224),             # ресайз каждой картинки до 224x224 (до формирования батча)
    # попробуйте другие стратегии ресайза:
    # item_tfms=Resize(224, method='squish')  — сплющивание (искажает пропорции)
    # item_tfms=Resize(224, method='pad')     — добавление чёрных полос
    batch_tfms=aug_transforms(         # аугментации — искусственно увеличиваем датасет
        max_rotate=10,                 # вращение до 10°
        max_zoom=1.1,                  # зум до 110%
        max_lighting=0.2,              # изменение яркости до 20%
        max_warp=0.1,                  # лёгкое искажение перспективы
        flip_vert=False,               # НЕ переворачивать вертикально (кошки не летают)
    ),
    bs=32,                             # batch size — сколько картинок за один шаг
)

# Выводим информацию о DataLoaders
print(f'Тренировочный набор: {len(dls.train.dataset)} изображений')
print(f'Валидационный набор: {len(dls.valid.dataset)} изображений')
print(f'Классы: {dls.vocab}')         # [False, True] = [собака, кошка]
print(f'Размер батча: {dls.bs}')
print(f'Количество батчей (train): {len(dls.train)}')
print(f'Количество батчей (valid): {len(dls.valid)}')

In [ ]:
# ============================================================
# Смотрим, что видит модель — батч тренировочных данных
# ============================================================
# show_batch() показывает один батч картинок С УЖЕ ПРИМЕНЁННЫМИ трансформациями
# Это именно то, что модель получает на вход
# Метка под каждой картинкой — это то, чему модель учится

dls.show_batch(max_n=8, nrows=2, figsize=(14, 6))

print('Обратите внимание:')
print('1. Все картинки приведены к размеру 224x224')
print('2. Некоторые картинки слегка повёрнуты/зумированы — это аугментации')
print('3. Метки: True = кошка, False = собака')
print('4. Это входные данные модели — она «видит» именно такие картинки')

In [ ]:
# ============================================================
# ШАГ 4b: Создаём Learner — объект обучения
# ============================================================
# vision_learner() создаёт Learner для vision-задач
# Он связывает: DataLoaders + Model + Loss + Optimizer + Metrics

# resnet18 — свёрточная сеть из 18 слоёв
# «Предобученная» = веса уже обучены на ImageNet (1.2 млн картинок, 1000 классов)
# Нам останется только «дообучить» последний слой под нашу задачу
#
# Альтернативы (попробуйте!):
#   resnet34  — 34 слоя, точнее, но в ~2x медленнее обучается
#   resnet50  — 50 слоёв, ещё точнее, ещё медленнее
#   squeezenet1_0 — маленькая модель для слабых GPU

learn = vision_learner(dls, resnet18, metrics=accuracy)

# Сколько параметров (весов) у модели?
# Параметр — это число, которое модель подстраивает в процессе обучения
total_params = sum(p.numel() for p in learn.model.parameters())
trainable_params = sum(p.numel() for p in learn.model.parameters() if p.requires_grad)

print(f'Архитектура: resnet18')
print(f'Всего параметров: {total_params:,}')
print(f'Из них обучаемых: {trainable_params:,}')
print(f'Замороженных (не обучаются на 1-й эпохе): {total_params - trainable_params:,}')
print()
print('11+ миллионов параметров — это «ручки», которые модель крутит в процессе обучения.')
print('Предобучение означает, что большинство «ручек» уже в хорошем положении!')
print('Нам нужно подкрутить только последние слои.')

In [ ]:
# ============================================================
# ШАГ 4c: Обучаем модель! (fine_tune)
# ============================================================
# fine_tune(3) — стратегия transfer learning в fastai:
#
# Фаза 1 (Эпоха 1): «заморозка»
#   - ВСЕ слои предобученной модели ЗАМОРОЖЕНЫ (не меняются)
#   - Обучается ТОЛЬКО последний слой (случайно инициализированный)
#   - Это быстро (мало параметров) и безопасно (не портим предобучение)
#
# Фаза 2 (Эпохи 2-3): «разморозка»
#   - Все слои РАЗМОРОЖЕНЫ
#   - Обучаются ВСЕ параметры, но с маленьким learning rate
#   - Это позволяет «подкрутить» ранние слои под нашу задачу
#
# Количество эпох (попробуйте разные!):
#   2 — мало, модель может недообучиться
#   3-5 — обычно оптимально
#   10+ — долго, может переобучиться (overfitting)

print('Начинаем обучение... (это займёт 2-5 минут)')
print('Наблюдайте за loss — он должен уменьшаться каждую эпоху!')
print()

learn.fine_tune(3)

print('\n✅ Обучение завершено!')

---
## Шаг 5. Оцениваем модель

Обучение — это половина дела. Вторая половина — понять, **насколько хорошо** модель работает.

Мы будем оценивать модель тремя способами:
1. **Loss-кривые** — как уменьшалась ошибка по эпохам (train vs valid)
2. **Confusion matrix** — где именно модель ошибается (кошка→собака или наоборот?)
3. **Топ ошибок** — самые «трудные» для модели картинки


In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Loss по эпохам
# ============================================================
# Loss = «ошибка» — чем меньше, тем лучше
# Train loss — ошибка на тренировочных данных (модель их ВИДИТ)
# Valid loss — ошибка на валидационных данных (модель их НЕ видит)
#
# Если train loss падает, а valid loss растёт — это ПЕРЕОБУЧЕНИЕ (overfitting)
# Модель запоминает тренировочные данные, но не обобщает

learn.recorder.plot_loss(figsize=(10, 5))
plt.title('Loss по эпохам (train vs valid)', fontsize=14, fontweight='bold')
plt.xlabel('Шаг обучения')
plt.ylabel('Loss (ошибка)')
plt.grid(True, alpha=0.3)

# Добавляем пояснения
plt.axhline(y=0.1, color='green', linestyle=':', alpha=0.5, label='Хороший loss')
plt.axhline(y=0.5, color='orange', linestyle=':', alpha=0.5, label='Средний loss')
plt.axhline(y=1.0, color='red', linestyle=':', alpha=0.5, label='Плохой loss')
plt.legend()
plt.show()

# Точность на валидации — главный показатель
val_loss, val_acc = learn.validate()
print(f'\nВалидационный loss: {val_loss:.4f}')
print(f'Точность на валидации: {val_acc:.2%}')
print()
if val_acc > 0.98:
    print('🔥 Отлично! Модель почти не ошибается.')
elif val_acc > 0.95:
    print('✅ Хорошо! Модель работает нормально.')
elif val_acc > 0.90:
    print('⚠️ Можно лучше. Попробуйте больше эпох или resnet34.')
else:
    print('❌ Модель недообучена. Проверьте данные и параметры.')

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Confusion Matrix (Матрица ошибок)
# ============================================================
# Confusion matrix показывает, КАК именно модель ошибается:
# - Диагональ = правильные предсказания
# - Вне диагонали = ошибки
#
#             Предсказана кошка  Предсказана собака
# Реально кошка      1400                50
# Реально собака      40               1350
#
# Если числа вне диагонали равны — модель одинаково ошибается в обе стороны
# Если одно число больше — есть систематическая ошибка

interp = ClassificationInterpretation.from_learner(learn)

# Рисуем матрицу ошибок
interp.plot_confusion_matrix(figsize=(5, 4), dpi=100)
plt.title('Матрица ошибок', fontsize=13)
plt.show()

# Читаем числа из матрицы
cm = interp.confusion_matrix()
print(f'Правильно классифицировано кошек: {cm[1][1]}')
print(f'Правильно классифицировано собак: {cm[0][0]}')
print(f'Кошка → собака (ошибка): {cm[1][0]}')
print(f'Собака → кошка (ошибка): {cm[0][1]}')
total_errors = cm[1][0] + cm[0][1]
total_samples = cm.sum()
print(f'\nВсего ошибок: {total_errors} из {total_samples} ({total_errors/total_samples:.1%})')

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Топ самых трудных картинок
# ============================================================
# plot_top_losses() показывает картинки, где модель ошиблась сильнее всего
# loss = насколько «удивилась» модель правильным ответом
# Больший loss = модель была уверена в неправильном ответе
#
# Под каждой картинкой:
#   предсказание | реальность | loss | вероятность предсказания

interp.plot_top_losses(9, nrows=3, figsize=(14, 10))
plt.suptitle('9 самых трудных картинок для модели', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Картинки сверху — самые трудные для модели.')
print('Обратите внимание: многие из них действительно неоднозначные!')
print('Некоторые кошки похожи на собак (и наоборот) — даже человеку сложно.')

---
## Шаг 6. Разбираем predict() — как модель принимает решение

Прежде чем упаковывать модель в сервис, нужно понять, **что именно** возвращает `predict()`. Без этого понимания невозможно написать надёжный API.

### Путь картинки через модель

```
Картинка → Ресайз 224x224 → Нормализация → Нейросеть → Логиты → Softmax → Вероятности → Решение
```

Каждый шаг — это конкретная математическая операция. Давайте разберём каждый.


In [ ]:
# ============================================================
# Разбираем predict() по шагам
# ============================================================

# Берём тестовую картинку
test_file = image_files[42]
test_img = PILImage.create(test_file)

print(f'Файл: {test_file.name}')
print(f'Размер: {test_img.size}')
print(f'Тип: {type(test_img)}')
print()

# ============================================================
# ШАГ 1: Препроцессинг (автоматический)
# ============================================================
# predict() делает это под капотом:
# - Ресайзит до 224x224 (как мы указали в item_tfms)
# - Нормализует пиксели: вычитает среднее ImageNet и делит на std
#   mean = [0.485, 0.456, 0.406]  std = [0.229, 0.224, 0.225]
# Это нужно, потому что модель обучалась на НОРМАЛИЗОВАННЫХ данных
# fastai запоминает трансформации из dls и применяет их автоматически

print('ШАГ 1: Препроцессинг (автоматический)')
print('  - Ресайз до 224x224')
print('  - Нормализация: pixel = (pixel - mean) / std')
print('  - Результат: тензор формы (1, 3, 224, 224)')
print('    1 = батч (одна картинка)')
print('    3 = каналы RGB')
print('    224x224 = пиксели')
print()

# ============================================================
# ШАГ 2: Прямой проход через нейросеть
# ============================================================
# 18 слоёв свёрток + пулинг + полносвязный слой
# Вход: (1, 3, 224, 224) -> Выход: (1, 2) — 2 «логита»
# Логит — «сырая» оценка класса (может быть любым числом)

print('ШАГ 2: Прямой проход через 18 слоёв')
print('  Вход: тензор (1, 3, 224, 224)')
print('  Выход: 2 логита (сырые оценки классов)')
print()

# ============================================================
# ШАГ 3: Softmax — превращение логитов в вероятности
# ============================================================
# softmax([a, b]) = [exp(a)/(exp(a)+exp(b)), exp(b)/(exp(a)+exp(b))]
# Сумма вероятностей ВСЕГДА = 1.0
# Чем больше разница между логитами — тем увереннее модель

print('ШАГ 3: Softmax — логиты → вероятности')
print('  P(кошка) + P(собака) = 1.0 (всегда!)')
print()

# ============================================================
# Вызываем predict() и смотрим результат
# ============================================================
pred_class, pred_idx, probs = learn.predict(test_img)

# pred_class — итоговый класс (True/False)
# pred_idx — индекс класса в dls.vocab
# probs — тензор вероятностей для ВСЕХ классов

true_label = '🐱 КОШКА' if is_cat(test_file.name) else '🐶 СОБАКА'
pred_label = '🐱 КОШКА' if pred_class else '🐶 СОБАКА'

print('РЕЗУЛЬТАТ:')
print(f'  Файл: {test_file.name}')
print(f'  Реальный класс: {true_label}')
print(f'  Предсказание:   {pred_label}')
print(f'  Индекс класса:  {pred_idx}')
print(f'  P(собака): {probs[0]:.4f} ({probs[0]:.2%})')
print(f'  P(кошка):  {probs[1]:.4f} ({probs[1]:.2%})')
print(f'  Уверенность: {max(probs):.2%}')
print()
if pred_label == true_label:
    print('✅ Модель права!')
else:
    print('❌ Модель ошиблась!')

In [ ]:
# ============================================================
# ВАЖНЫЙ ЭКСПЕРИМЕНТ: что если подать «мусор»?
# ============================================================
# Модель ВСЕГДА даст ответ — даже если подать шум или не-фото
# Это фундаментальная проблема для продакшена!

# Создаём случайный шум (224x224x3 случайных пикселей)
random_noise = torch.randint(0, 256, (224, 224, 3), dtype=torch.uint8)
random_pil = PILImage.create(random_noise.numpy())

# Показываем «мусор»
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Случайный шум
axes[0].imshow(random_noise.numpy())
axes[0].set_title('Случайный шум (не кошка и не собака)')
axes[0].axis('off')

# Предсказание для шума
pred_class, pred_idx, probs = learn.predict(random_pil)
pred_label = '🐱 КОШКА' if pred_class else '🐶 СОБАКА'

# Визуализация вероятностей
axes[1].barh(['Собака', 'Кошка'], [probs[0], probs[1]], 
             color=['#4ECDC4', '#FF6B6B'])
axes[1].set_xlim(0, 1)
axes[1].set_xlabel('Вероятность')
axes[1].set_title(f'Модель говорит: {pred_label} ({max(probs):.1%})')

plt.tight_layout()
plt.show()

print(f'\nМодель ответила: {pred_label} с уверенностью {max(probs):.2%}')
print()
print('⚠️ МОДЕЛЬ ВСЕГДА ЧТО-ТО ОТВЕТИТ!')
print('Она не умеет говорить «я не знаю».')
print('Это фундаментальная проблема, которую нужно решать в продакшене.')
print()
print('🔧 Решение: добавить ПОРОГ УВЕРЕННОСТИ.')
print('Если max(probs) < порог — отвечаем «не уверен» вместо случайного класса.')
print('Именно это мы реализуем в FastAPI-сервисе!')

---
## Шаг 7. Интерактивный тестер модели

Прежде чем упаковывать модель в сервис, давайте поиграемся с ней прямо в блокноте.

**Попробуйте:**
1. Листайте датасет слайдером — смотрите, где модель ошибается
2. Меняйте порог уверенности — наблюдайте, как меняются решения
3. Найдите картинки, где модель ошибается — почему?


In [ ]:
# ============================================================
# ИНТЕРАКТИВНЫЙ ТЕСТЕР: листаем датасет + порог уверенности
# ============================================================
# Два слайдера:
# 1. Индекс картинки — выбираем изображение из датасета
# 2. Порог уверенности — если max(prob) < порог, модель «не уверена»

idx_slider = widgets.IntSlider(
    value=0, min=0, max=len(image_files)-1, step=1,
    description='Картинка #:',
    layout=widgets.Layout(width='500px')
)

threshold_slider = widgets.FloatSlider(
    value=0.7, min=0.5, max=0.99, step=0.01,
    description='Порог:',
    layout=widgets.Layout(width='300px'),
    style={'description_width': 'initial'}
)

output = widgets.Output()

def test_model(idx, threshold):
    """Интерактивный тест: картинка + предсказание + порог.
    
    Показывает:
    - Изображение с реальной и предсказанной меткой
    - Столбчатую диаграмму вероятностей
    - Линию порога на диаграмме
    - Решение: уверена ли модель?
    
    Args:
        idx: индекс картинки в image_files
        threshold: порог уверенности (0.5-0.99)
    """
    with output:
        clear_output(wait=True)
        f = image_files[idx]
        img = PILImage.create(f)
        true_label = '🐱 КОШКА' if is_cat(f.name) else '🐶 СОБАКА'
        
        # Получаем предсказание
        pred_class, pred_idx, probs = learn.predict(img)
        pred_label = '🐱 КОШКА' if pred_class else '🐶 СОБАКА'
        confidence = max(probs).item()
        
        # Определяем решение с учётом порога
        if confidence >= threshold:
            decision = pred_label
            decision_color = 'green' if pred_label == true_label else 'red'
        else:
            decision = '❓ НЕ УВЕРЕНА'
            decision_color = 'orange'
        
        # Рисуем: картинка + вероятности
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Картинка
        axes[0].imshow(img)
        axes[0].axis('off')
        is_correct = pred_label == true_label
        axes[0].set_title(f'Реальность: {true_label}\nПредсказание: {pred_label} ({confidence:.1%})',
                         color='green' if is_correct else 'red', fontsize=11)
        
        # Вероятности
        bar_colors = ['#4ECDC4', '#FF6B6B']
        bars = axes[1].barh(['Собака', 'Кошка'], [probs[0], probs[1]], color=bar_colors)
        axes[1].axvline(x=threshold, color='red', linestyle='--', linewidth=2, 
                       label=f'Порог = {threshold:.2f}')
        axes[1].set_xlim(0, 1)
        axes[1].set_xlabel('Вероятность')
        axes[1].set_title('Вероятности классов')
        axes[1].legend()
        
        # Подписи на столбцах
        for bar, prob in zip(bars, [probs[0], probs[1]]):
            axes[1].text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                        f'{prob:.2%}', va='center', fontsize=11, fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        # Решение
        print(f'Решение: {decision}')
        print(f'Уверенность: {confidence:.2%} (порог: {threshold:.2%})')
        if confidence < threshold:
            print('⚠️ Модель не уверена! В продакшене вернули бы «не знаю».')

# Связываем слайдеры с функцией
widgets.interact(test_model, idx=idx_slider, threshold=threshold_slider)
display(output)

### Поиграйте с порогом!

- **Порог 0.50** — модель почти всегда «уверена», но иногда ошибается. Много ложных предсказаний.
- **Порог 0.70** — разумный компромисс. Немного отказов, мало ошибок.
- **Порог 0.95** — модель часто «не уверена». Мало ошибок, но много отказов.
- **Порог 0.99** — модель почти никогда не уверена. Практически нет предсказаний.

> **Золотое правило:** порог — это баланс между **точностью** (мало ошибок) и **полнотой** (мало отказов).

---

---
## Шаг 8. Сохраняем и загружаем модель

Обученную модель нужно сохранить в файл. Зачем?
- **Не обучать каждый раз заново** — обучение занимает минуты/часы
- **Деплой** — на сервере нужна модель из файла, не процесс обучения
- **Воспроизводимость** — одна и та же модель на разных машинах

Есть два способа сохранить модель в fastai/PyTorch:


In [ ]:
# ============================================================
# Способ 1: learn.export() — РЕКОМЕНДУЕТСЯ для продакшена
# ============================================================
# learn.export() сохраняет ВООБЩЕ ВСЁ:
# - Архитектуру модели (какие слои, в каком порядке)
# - Веса (обученные параметры)
# - Препроцессинг (трансформации из dls — ресайз, нормализация)
# - Словарь классов (dls.vocab — что такое True/False)
# - Optimizer state (для продолжения обучения)
#
# В результате: load_learner() полностью восстанавливает модель
# БЕЗ какого-либо дополнительного кода!

learn.export('cat_dog_model.pkl')

import os
model_path = 'cat_dog_model.pkl'
size_mb = os.path.getsize(model_path) / 1e6
print(f'✅ Модель сохранена: {model_path} ({size_mb:.1f} МБ)')
print(f'   Формат: pickle (сериализованный Python-объект)')

# ============================================================
# Загружаем обратно и проверяем
# ============================================================
loaded_learn = load_learner(model_path)

# Проверяем: совпадают ли результаты?
test_img = PILImage.create(image_files[200])

# Оригинальная модель
p1, _, pr1 = learn.predict(test_img)
# Загруженная модель
p2, _, pr2 = loaded_learn.predict(test_img)

label1 = 'кошка' if p1 else 'собака'
label2 = 'кошка' if p2 else 'собака'

print(f'\nОригинальная модель:  {label1} (prob={max(pr1):.4f})')
print(f'Загруженная модель:   {label2} (prob={max(pr2):.4f})')
print(f'Результаты идентичны: {"✅ ДА" if p1 == p2 else "❌ НЕТ"}')

In [ ]:
# ============================================================
# Способ 2: torch.save(state_dict) — НИЗКОУРОВНЕВЫЙ
# ============================================================
# state_dict — это словарь {имя_параметра: тензор_весов}
# Сохраняет ТОЛЬКО веса (числа), без архитектуры и препроцессинга
#
# Плюсы:
#   - Меньше размер файла
#   - Более переносимый формат (работает между версиями)
#   - Безопаснее (pickle может выполнять произвольный код)
#
# Минусы:
#   - Нужно ЗНАТЬ архитектуру модели при загрузке
#   - Нужно САМОМУ делать препроцессинг (ресайз, нормализация)
#   - Нужно САМОМУ применять softmax к логитам

torch.save(learn.model.state_dict(), 'cat_dog_weights.pth')

weights_mb = os.path.getsize('cat_dog_weights.pth') / 1e6
print(f'✅ Веса сохранены: cat_dog_weights.pth ({weights_mb:.1f} МБ)')
print(f'   Разница в размере: {size_mb - weights_mb:.1f} МБ')
print(f'   (.pkl хранит ещё архитектуру + препроцессинг + словарь классов)')

# Чтобы загрузить через state_dict, нужно:
# 1. Создать модель с ТАКОЙ ЖЕ архитектурой
# 2. Загрузить веса: model.load_state_dict(torch.load('weights.pth'))
# 3. Самому делать препроцессинг (ресайз, нормализация)
# 4. Самому применять softmax к логитам

print('\n⚠️ При загрузке через state_dict нужно:')
print('   1. Создать модель с ТАКОЙ ЖЕ архитектурой')
print('   2. Загрузить веса: model.load_state_dict(torch.load(...))')
print('   3. Самому делать препроцессинг (ресайз 224, нормализация ImageNet)')
print('   4. Самому применять softmax к выходу')
print('\nВ этом курсе мы используем learn.export() — проще и надёжнее!')
print('Но в «серьёзном» продакшене часто используют state_dict.')

### Сравнение способов

| | `learn.export()` | `torch.save(state_dict)` |
|---|---|---|
| **Что сохраняет** | Всё (архитектура + веса + препроцессинг + словарь классов) | Только веса (числа) |
| **Размер файла** | Больше (~47 МБ) | Меньше (~45 МБ) |
| **Загрузка** | `load_learner('model.pkl')` — одна строка | Нужно воссоздать архитектуру + препроцессинг |
| **Безопасность** | pickle (потенциально небезопасно) | Безопасно (только числа) |
| **Когда использовать** | Прототипы, Colab, демо | Продакшен, мобильные приложения |

---

---
## Шаг 9. Блокнот vs продакшен

Это **самая важная часть** ноутбука. Прежде чем писать код сервиса, нужно понять, чем блокнот отличается от «настоящего» приложения.

<img src="https://sfile.chatglm.cn/images-ppt/fecac9c1b165.png" width="500" align="center">

*Слева: мир блокнота — вы контролируете всё. Справа: мир продакшена — пользователи непредсказуемы.*


In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Путь от блокнота к продакшену
# ============================================================
# Рисуем две параллельные «дорожки» — блокнот и продакшен

import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# ---- БЛОКНОТ (слева) ----
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('📓 Блокнот', fontsize=18, fontweight='bold', pad=15)

# Блоки блокнота
notebook_steps = [
    (1, 8.5, 'Ваши данные\n(вы контролируете)', 'lightblue', 8, 1.2),
    (1, 7.0, 'Модель в памяти\n(пока открыт ноутбук)', 'lightyellow', 8, 1.2),
    (1, 5.5, 'Результат в ячейке\n(видите вы)', 'lightgreen', 8, 1.2),
    (1, 4.0, 'Ошибка = стоп\n(чините вручную)', 'lightsalmon', 8, 1.2),
    (1, 2.0, 'Один пользователь\n(вы сами)', 'plum', 8, 1.2),
]

for x, y, text, color, w, h in notebook_steps:
    ax.add_patch(plt.Rectangle((x, y-h/2), w, h, fc=color, ec='gray', lw=1.5, 
                               alpha=0.8, zorder=2))
    ax.text(x + w/2, y, text, ha='center', va='center', fontsize=10, fontweight='bold')

# Стрелки между блоками
for i in range(len(notebook_steps)-1):
    y_from = notebook_steps[i][1] - notebook_steps[i][5]/2
    y_to = notebook_steps[i+1][1] + notebook_steps[i+1][5]/2
    ax.annotate('', xy=(5, y_to), xytext=(5, y_from),
               arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

# ---- ПРОДАКШЕН (справа) ----
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('🚀 Продакшен', fontsize=18, fontweight='bold', pad=15)

prod_steps = [
    (1, 8.5, 'Чужие данные\n(что угодно! PDF, мусор, 100 МБ)', '#FFB3BA', 8, 1.2),
    (1, 7.0, 'Модель из файла\n(загружается при старте)', 'lightyellow', 8, 1.2),
    (1, 5.5, 'JSON-ответ\n(видит программа)', '#BAFFC9', 8, 1.2),
    (1, 4.0, 'Ошибка = обработка\n(HTTP-код + сообщение)', '#BAE1FF', 8, 1.2),
    (1, 2.0, 'Тысячи пользователей\n(одновременно!)', '#FFFFBA', 8, 1.2),
]

for x, y, text, color, w, h in prod_steps:
    ax.add_patch(plt.Rectangle((x, y-h/2), w, h, fc=color, ec='gray', lw=1.5,
                               alpha=0.8, zorder=2))
    ax.text(x + w/2, y, text, ha='center', va='center', fontsize=10, fontweight='bold')

# Стрелки между блоками
for i in range(len(prod_steps)-1):
    y_from = prod_steps[i][1] - prod_steps[i][5]/2
    y_to = prod_steps[i+1][1] + prod_steps[i+1][5]/2
    ax.annotate('', xy=(5, y_to), xytext=(5, y_from),
               arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

plt.suptitle('Блокнот → Продакшен: что меняется?', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print('Ключевая мысль: в продакшене вы НЕ контролируете входные данные!')
print('Поэтому НУЖНА валидация, обработка ошибок и пороги уверенности.')

In [ ]:
# ============================================================
# ИНТЕРАКТИВ: Таблица различий — аккордеон
# ============================================================
# Раскройте каждый пункт, чтобы увидеть подробное сравнение

aspects = widgets.Accordion(children=[
    widgets.HTML('''
        <h4>📥 Входные данные</h4>
        <table style="width:100%">
        <tr><td><b>В блокноте:</b></td><td>Вы контролируете входные данные. 
        Если картинка не открывается — вы сами её замените.</td></tr>
        <tr><td><b>В продакшене:</b></td><td>Пользователь может отправить что угодно — 
        PDF, пустой файл, 100 МБ картинку, текстовый файл. 
        <b>Нужна валидация!</b></td></tr>
        </table>
    '''),
    widgets.HTML('''
        <h4>⏳ Жизненный цикл модели</h4>
        <table style="width:100%">
        <tr><td><b>В блокноте:</b></td><td>Модель живёт, пока работает ядро. 
        Перезапуск ядра = модель пропала.</td></tr>
        <tr><td><b>В продакшене:</b></td><td>Модель загружается из файла при старте сервера. 
        Сервер работает 24/7, модель всегда доступна.</td></tr>
        </table>
    '''),
    widgets.HTML('''
        <h4>📤 Формат результата</h4>
        <table style="width:100%">
        <tr><td><b>В блокноте:</b></td><td>Результат видите вы в ячейке — 
        картинка, текст, таблица.</td></tr>
        <tr><td><b>В продакшене:</b></td><td>Результат отдаётся как JSON через HTTP. 
        Клиент — другая программа, не человек. 
        Нужен чёткий формат ответа.</td></tr>
        </table>
    '''),
    widgets.HTML('''
        <h4>⚠️ Обработка ошибок</h4>
        <table style="width:100%">
        <tr><td><b>В блокноте:</b></td><td>Ошибка = стоп. Чините вручную, 
        перезапускаете ячейку.</td></tr>
        <tr><td><b>В продакшене:</b></td><td>Ошибка = ответ с HTTP-кодом 
        (400, 422, 500). Сервер НЕ падает, 
        продолжает обслуживать других пользователей.</td></tr>
        </table>
    '''),
    widgets.HTML('''
        <h4>🔒 Безопасность</h4>
        <table style="width:100%">
        <tr><td><b>В блокноте:</b></td><td>Вы единственный пользователь. 
        Нет проблем с безопасностью.</td></tr>
        <tr><td><b>В продакшене:</b></td><td>Нужна авторизация, ограничение 
        размера файлов, защита от DDoS, HTTPS.</td></tr>
        </table>
    '''),
    widgets.HTML('''
        <h4>📊 Мониторинг</h4>
        <table style="width:100%">
        <tr><td><b>В блокноте:</b></td><td>Вы видите всё своими глазами. 
        Если что-то не так — замечаете сразу.</td></tr>
        <tr><td><b>В продакшене:</b></td><td>Нужны логи, метрики, алерты. 
        Если модель стала хуже работать — нужно узнать об этом 
        ДО жалоб пользователей.</td></tr>
        </table>
    '''),
])

aspects_titles = ['📥 Входные данные', '⏳ Жизненный цикл', '📤 Формат результата',
                  '⚠️ Обработка ошибок', '🔒 Безопасность', '📊 Мониторинг']
for i, title in enumerate(aspects_titles):
    aspects.set_title(i, title)

display(aspects)

---
## Шаг 10. Пишем FastAPI-сервис

<img src="https://sfile.chatglm.cn/images-ppt/790d390f0534.jpg" width="400" align="center">

*FastAPI — современный фреймворк для создания API на Python. Быстрый, с автоматической документацией, валидацией данных.*

Теперь мы напишем **настоящий API-сервис**, который:
1. Загружает модель из файла при старте
2. Принимает картинку через HTTP POST
3. Возвращает предсказание в формате JSON
4. Обрабатывает ошибки (некорректный файл, слишком большой, и т.д.)
5. Применяет порог уверенности

Мы создадим **3 файла**:
- `model_service.py` — логика модели (загрузка, предсказание)
- `main.py` — HTTP-сервер (маршруты, валидация, ошибки)
- `requirements.txt` — зависимости

### Принцип разделения ответственности

```
model_service.py  →  знает ТОЛЬКО про модель (загрузка, предсказание)
main.py           →  знает ТОЛЬКО про HTTP (запросы, ответы, маршруты)
```

Если завтра мы заменим fastai на PyTorch — меняем только `model_service.py`.
Если заменим FastAPI на Flask — меняем только `main.py`.


In [ ]:
# ============================================================
# Подготовка: создаём директорию и копируем модель
# ============================================================

import os, shutil

# Создаём папку app/ — там будут жить файлы сервиса
os.makedirs('app', exist_ok=True)

# Копируем обученную модель в app/
# Сервис будет загружать модель из этой папки
shutil.copy('cat_dog_model.pkl', 'app/cat_dog_model.pkl')

print('✅ app/ создана, модель скопирована')
print(f'   Размер модели: {os.path.getsize("app/cat_dog_model.pkl")/1e6:.1f} МБ')

In [ ]:
# ============================================================
# ФАЙЛ 1: model_service.py — логика модели
# ============================================================
# Этот файл инкапсулирует ВСЮ работу с моделью:
# - Загрузка из файла
# - Предсказание
# - Проверка уверенности
# - Обработка ошибок
#
# Принцип: если завтра заменим fastai на чистый PyTorch,
# нужно переписать ТОЛЬКО этот файл. main.py не меняется!

model_service_code = '''"""
model_service.py — инкапсулирует логику загрузки модели и предсказаний.

Зачем отдельный файл? Принцип разделения ответственности:
- Этот файл знает только про модель (загрузка, предсказание)
- main.py знает только про HTTP (запросы, ответы, маршруты)

Если завтра заменим fastai на PyTorch — меняем только этот файл.
"""

import io                          # для работы с байтами в памяти
import torch                       # PyTorch — для работы с тензорами
from fastai.vision.all import load_learner, PILImage  # загрузка модели и картинки
from PIL import Image              # работа с изображениями
from pathlib import Path           # работа с путями к файлам


# ========== КОНСТАНТЫ ==========
# Все настраиваемые параметры — в одном месте,
# чтобы не искать их по всему коду

MODEL_PATH = Path("cat_dog_model.pkl")   # путь к файлу модели
CONFIDENCE_THRESHOLD = 0.7               # порог уверенности (70%)
MAX_IMAGE_SIZE = 10 * 1024 * 1024        # макс. размер файла (10 МБ)
ALLOWED_TYPES = {"image/jpeg", "image/png", "image/webp"}  # допустимые форматы


# ========== ГЛОБАЛЬНАЯ ПЕРЕМЕННАЯ ==========
# Модель загружается ОДИН раз при старте сервера
# и живёт в памяти, пока сервер работает
# Глобальная переменная — простой способ (для продакшена лучше dependency injection)
_model = None


def load_model():
    """Загружает модель из файла (один раз при старте сервера).
    
    Возвращает объект Learner, который:
    - Содержит архитектуру модели + обученные веса
    - Знает, как делать препроцессинг (ресайз, нормализация)
    - Знает словарь классов (кошка/собака)
    
    Returns:
        Learner: загруженная модель fastai
    """
    global _model
    if _model is None:
        # load_learner() восстанавливает ВСЁ из .pkl файла:
        # - архитектуру (какие слои)
        # - веса (обученные параметры)
        # - препроцессинг (трансформации из dls)
        # - словарь классов (dls.vocab)
        _model = load_learner(MODEL_PATH)
        print(f"Модель загружена из {MODEL_PATH}")
        print(f"Классы: {_model.dls.vocab}")
    return _model


def predict(image_bytes: bytes) -> dict:
    """Делает предсказание по байтам изображения.
    
    ПОЛНЫЙ ПУТЬ картинки через эту функцию:
    1. Проверяем размер файла
    2. Открываем изображение из байтов
    3. Делаем предсказание через модель
    4. Проверяем уверенность (порог)
    5. Возвращаем структурированный результат
    
    Args:
        image_bytes: сырые байты картинки (из HTTP-запроса)
    
    Returns:
        dict с ключами:
            - class: str — "кот" или "собака" (или "не уверен")
            - confidence: float — уверенность (0.0-1.0)
            - is_reliable: bool — достаточно ли уверенности
            - all_probs: dict — вероятности всех классов
    """
    # ---- Шаг 1: Проверяем размер ----
    if len(image_bytes) > MAX_IMAGE_SIZE:
        return {
            "error": f"Файл слишком большой: {len(image_bytes)/1e6:.1f} МБ (максимум {MAX_IMAGE_SIZE/1e6:.0f} МБ)",
            "class": None,
            "confidence": 0.0,
            "is_reliable": False,
        }
    
    # ---- Шаг 2: Открываем изображение ----
    try:
        img = PILImage.create(io.BytesIO(image_bytes))
    except Exception as e:
        return {
            "error": f"Не удалось открыть изображение: {str(e)}",
            "class": None,
            "confidence": 0.0,
            "is_reliable": False,
        }
    
    # ---- Шаг 3: Предсказание ----
    model = load_model()
    pred_class, pred_idx, probs = model.predict(img)
    
    # pred_class — True/False (кошка/собака)
    # probs — тензор вероятностей: [P(собака), P(кошка)]
    confidence = max(probs).item()    # наибольшая вероятность
    class_name = "кот" if pred_class else "собака"
    
    # ---- Шаг 4: Проверяем порог уверенности ----
    is_reliable = confidence >= CONFIDENCE_THRESHOLD
    
    # ---- Шаг 5: Формируем результат ----
    result = {
        "class": class_name if is_reliable else "не уверен",
        "confidence": round(confidence, 4),       # округляем до 4 знаков
        "is_reliable": is_reliable,
        "all_probs": {
            "собака": round(probs[0].item(), 4),
            "кот": round(probs[1].item(), 4),
        }
    }
    
    return result
'''

# Записываем файл
with open('app/model_service.py', 'w', encoding='utf-8') as f:
    f.write(model_service_code)

print('✅ app/model_service.py создан')
print(f'   Размер: {len(model_service_code)} символов')
print(f'   Функции: load_model(), predict()')
print(f'   Константы: MODEL_PATH, CONFIDENCE_THRESHOLD, MAX_IMAGE_SIZE, ALLOWED_TYPES')

In [ ]:
# ============================================================
# ФАЙЛ 2: main.py — API-сервер
# ============================================================
# Этот файл отвечает за HTTP-интерфейс:
# - Маршруты (GET /, GET /info, POST /predict)
# - Валидация входных данных (тип файла, размер)
# - Форматирование ответов (JSON)
# - Обработка ошибок (HTTP-коды 400, 422, 500)

main_code = '''"""
main.py — FastAPI-сервер для классификации кошек/собак.

Эндпоинты (маршруты):
- GET  /         -> health check (сервер работает?)
- GET  /info     -> информация о модели
- POST /predict  -> классификация изображения
"""

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
import time

# Импортируем логику модели из отдельного файла
from model_service import (
    load_model,                  # функция загрузки модели
    predict,                     # функция предсказания
    MODEL_PATH,                  # путь к файлу модели
    CONFIDENCE_THRESHOLD,        # порог уверенности
    MAX_IMAGE_SIZE,              # макс. размер файла
    ALLOWED_TYPES,               # допустимые типы
)


# ============================================================
# Создаём приложение FastAPI
# ============================================================
# title, description, version — отображаются в автоматической документации
# (доступна по адресу /docs после запуска сервера)
app = FastAPI(
    title="Cat/Dog Classifier API",
    description="API для классификации изображений: кошка или собака. "
                "Обучено на Oxford-IIIT Pet Dataset с помощью fastai + resnet18.",
    version="1.0.0",
)


# ============================================================
# Событие запуска: загружаем модель
# ============================================================
# @app.on_event("startup") — функция выполняется ОДИН раз при старте сервера
# Модель загружается в память и остаётся там до остановки
@app.on_event("startup")
async def startup():
    """Загружает модель при старте сервера."""
    load_model()
    print("✅ Модель загружена и готова к работе!")


# ============================================================
# ЭНДПОИНТ 1: Health Check
# ============================================================
# GET / — самый простой эндпоинт
# Используется для проверки: жив ли сервер?
# Мониторинг (Kubernetes, Docker) пингует этот эндпоинт
@app.get("/")
async def root():
    """Health check — сервер работает?"""
    return {"status": "ok", "message": "Cat/Dog Classifier API работает!"}


# ============================================================
# ЭНДПОИНТ 2: Информация о модели
# ============================================================
# GET /info — возвращает метаданные о модели
# Полезно для отладки и документации
@app.get("/info")
async def info():
    """Информация о модели и настройках."""
    model = load_model()
    return {
        "model_type": "fastai vision_learner (resnet18)",
        "classes": list(map(str, model.dls.vocab)),  # [False, True] -> ['False', 'True']
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "max_image_size_mb": MAX_IMAGE_SIZE / 1e6,
        "allowed_types": list(ALLOWED_TYPES),
    }


# ============================================================
# ЭНДПОИНТ 3: Классификация изображения
# ============================================================
# POST /predict — главный эндпоинт
# Принимает файл (картинку) через multipart/form-data
# Возвращает JSON с результатом классификации
@app.post("/predict")
async def predict_endpoint(file: UploadFile = File(...)):
    """Классификация загруженного изображения.
    
    Args:
        file: загруженный файл (UploadFile — специальный тип FastAPI)
    
    Returns:
        JSON с ключами: class, confidence, is_reliable, all_probs
    
    Raises:
        HTTPException 400: файл не является изображением
        HTTPException 500: внутренняя ошибка сервера
    """
    # ---- Валидация типа файла ----
    # content_type — MIME-тип загруженного файла
    # Если это не картинка — сразу отказываем
    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(
            status_code=400,     # 400 = Bad Request (ошибка клиента)
            detail=f"Неподдерживаемый тип файла: {file.content_type}. "
                   f"Допустимые: {ALLOWED_TYPES}"
        )
    
    # ---- Читаем байты файла ----
    # await — асинхронное чтение (не блокирует сервер)
    image_bytes = await file.read()
    
    # ---- Вызываем модель ----
    try:
        result = predict(image_bytes)
    except Exception as e:
        # Если модель упала — это ошибка СЕРВЕРА (500)
        raise HTTPException(
            status_code=500,
            detail=f"Ошибка при предсказании: {str(e)}"
        )
    
    # ---- Проверяем ошибку валидации (из model_service) ----
    if result.get("error"):
        raise HTTPException(
            status_code=400,
            detail=result["error"]
        )
    
    # ---- Добавляем метаданные ----
    result["filename"] = file.filename
    result["content_type"] = file.content_type
    
    return result
'''

# Записываем файл
with open('app/main.py', 'w', encoding='utf-8') as f:
    f.write(main_code)

print('✅ app/main.py создан')
print(f'   Размер: {len(main_code)} символов')
print(f'   Эндпоинты: GET /, GET /info, POST /predict')
print(f'   Обработка ошибок: HTTP 400, 422, 500')

In [ ]:
# ============================================================
# ФАЙЛ 3: requirements.txt
# ============================================================
# requirements.txt — список зависимостей проекта
# pip install -r requirements.txt установит всё необходимое
# Версии указаны с минимальным ограничением (>=), чтобы 
# обеспечить совместимость

requirements = '''fastai>=2.7
fastapi>=0.100
uvicorn>=0.23
python-multipart>=0.0.6
Pillow>=9.0
torch>=1.13
torchvision>=0.14
'''

with open('app/requirements.txt', 'w') as f:
    f.write(requirements.strip())

print('✅ app/requirements.txt создан')
print()
print('Структура проекта:')
for root, dirs, files in os.walk('app'):
    level = root.replace('app', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in files:
        size = os.path.getsize(os.path.join(root, file))
        if size > 1e6:
            size_str = f'{size/1e6:.1f} МБ'
        else:
            size_str = f'{size} Б'
        print(f'{subindent}{file} ({size_str})')

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Архитектура сервиса
# ============================================================
# Показываем, как данные проходят через систему
# от пользователя до модели и обратно

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')

# ---- Компоненты ----
components = [
    (0.3, 2.5, 2.0, 2.0, 'Пользователь\n(клиент)', 'lightblue', 11),
    (3.5, 2.5, 2.5, 2.0, 'FastAPI\n(main.py)', 'lightyellow', 11),
    (7.0, 2.5, 2.5, 2.0, 'model_service.py\n(логика модели)', '#BAFFC9', 10),
    (10.5, 2.5, 2.5, 2.0, 'cat_dog_model.pkl\n(файл модели)', 'plum', 10),
]

for x, y, w, h, text, color, fontsize in components:
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=color, ec='steelblue', lw=2, alpha=0.8, zorder=2))
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize, fontweight='bold')

# ---- Стрелки с подписями ----
# Пользователь -> FastAPI
ax.annotate('', xy=(3.5, 3.5), xytext=(2.3, 3.5),
           arrowprops=dict(arrowstyle='->', lw=2.5, color='green'))
ax.text(2.9, 4.7, 'POST /predict\n(файл.jpg)', ha='center', fontsize=9, color='green', fontweight='bold')

# FastAPI -> model_service
ax.annotate('', xy=(7.0, 3.5), xytext=(6.0, 3.5),
           arrowprops=dict(arrowstyle='->', lw=2.5, color='blue'))
ax.text(6.5, 4.7, 'predict(bytes)', ha='center', fontsize=9, color='blue', fontweight='bold')

# model_service -> модель
ax.annotate('', xy=(10.5, 3.5), xytext=(9.5, 3.5),
           arrowprops=dict(arrowstyle='->', lw=2.5, color='purple'))
ax.text(10.0, 4.7, 'load_model()', ha='center', fontsize=9, color='purple', fontweight='bold')

# Обратные стрелки
ax.annotate('', xy=(6.0, 3.0), xytext=(7.0, 3.0),
           arrowprops=dict(arrowstyle='->', lw=2, color='orange', linestyle='dashed'))
ax.text(6.5, 2.2, 'результат\ndict', ha='center', fontsize=9, color='orange')

ax.annotate('', xy=(2.3, 3.0), xytext=(3.5, 3.0),
           arrowprops=dict(arrowstyle='->', lw=2, color='red', linestyle='dashed'))
ax.text(2.9, 2.2, 'JSON\n{class, confidence}', ha='center', fontsize=9, color='red')

# ---- Легенда ----
ax.text(7, 6.5, 'Архитектура FastAPI-сервиса', ha='center', fontsize=16, fontweight='bold')
ax.text(7, 6.0, 'Сплошные стрелки = запрос →  Пунктирные = ответ ←', ha='center', fontsize=10, color='gray')

# Этапы обработки
steps_text = [
    '1. Валидация типа файла',
    '2. Чтение байтов',
    '3. Предсказание модели',
    '4. Проверка порога',
    '5. Формирование JSON',
]
for i, step in enumerate(steps_text):
    ax.text(0.5, 1.0 - i*0.25, step, fontsize=9, color='steelblue',
           bbox=dict(boxstyle='round,pad=0.2', fc='lightyellow', ec='steelblue', alpha=0.7))

plt.tight_layout()
plt.show()

---
## Шаг 11. Запускаем и тестируем сервис

<img src="https://sfile.chatglm.cn/images-ppt/fecac9c1b165.png" width="400" align="center">

Теперь самое интересное — запустим наш сервис и протестируем его!

### Что будем тестировать

| Тест | Что проверяем | Ожидаемый результат |
|------|--------------|-------------------|
| 0 | model_service без сервера | Функция predict() работает |
| 1 | Health check (GET /) | 200 OK |
| 2 | Информация о модели (GET /info) | JSON с метаданными |
| 3 | Классификация картинки (POST /predict) | JSON с предсказанием |
| 4 | Серия картинок — визуальная сетка | 3x3 картинок с результатами |
| 5 | «Плохой» запрос — текст вместо картинки | 400 Bad Request |
| 6 | Слишком большой файл | 400 Bad Request |


In [ ]:
# ============================================================
# ТЕСТ 0: model_service без сервера (юнит-тест)
# ============================================================
# Сначала проверяем логику модели БЕЗ HTTP
# Это называется «юнит-тест» — тестируем одну «единицу» (функцию)
# Если predict() не работает — сервер тоже не будет работать

import sys
sys.path.insert(0, 'app')  # добавляем app/ в путь импорта

from model_service import load_model, predict

# Загружаем модель
model = load_model()
print(f'Тип модели: {type(model)}')
print(f'Классы: {model.dls.vocab}')
print(f'Размер входа: {model.dls.train.after_item.size}')
print()

# Тест 1: реальная картинка
with open(image_files[50], 'rb') as f:
    real_result = predict(f.read())

print('✅ Реальная картинка:')
for key, value in real_result.items():
    print(f'   {key}: {value}')

print()

# Тест 2: «мусор» вместо картинки
bad_result = predict(b'this is not an image at all')
print('✅ Мусор на входе:')
for key, value in bad_result.items():
    print(f'   {key}: {value}')

print('\nМодель НЕ упала! Вернула понятную ошибку вместо исключения.')

In [ ]:
# ============================================================
# Запускаем сервер!
# ============================================================
# uvicorn — ASGI-сервер, который запускает FastAPI-приложение
# Запускаем в ФОНЕ (subprocess), чтобы продолжать работать в блокноте

import subprocess, time

# Убиваем предыдущий сервер (если был запущен)
# pkill ищет процесс по имени и убивает его
!pkill -f 'uvicorn.*main:app' 2>/dev/null || true
time.sleep(1)  # даём время на завершение

# Запускаем сервер в фоне
# subprocess.Popen = запустить процесс, не дожидаясь завершения
# cwd='app' = запустить из папки app/ (чтобы main.py нашёл модель)
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='app',                    # рабочая директория
    stdout=subprocess.PIPE,       # перехватываем stdout
    stderr=subprocess.PIPE        # перехватываем stderr
)

print('Запуск сервера...')
time.sleep(5)  # даём время на загрузку модели (~5 секунд)

# Проверяем, что сервер запустился
import httpx  # HTTP-клиент для тестирования
try:
    r = httpx.get('http://localhost:8000/', timeout=5)
    print(f'✅ Сервер работает! Статус: {r.status_code}')
    print(f'   Ответ: {r.json()}')
except Exception as e:
    print(f'❌ Сервер не отвечает: {e}')
    print('   Проверьте вывод ошибки ниже:')
    print(server.stderr.read().decode())

In [ ]:
# ============================================================
# ТЕСТ 1: Health check — проверяем, что сервер жив
# ============================================================
# GET / — самый простой запрос
# Если он работает — сервер запущен и отвечает

r = httpx.get('http://localhost:8000/')

print(f'Метод: GET /')
print(f'Статус: {r.status_code} (200 = OK)')
print(f'Ответ: {r.json()}')
print()
print('Интерпретация:')
print('  200 = сервер работает, модель загружена')
print('  Если нет ответа = сервер не запустился (проверьте ошибку выше)')

In [ ]:
# ============================================================
# ТЕСТ 2: Информация о модели
# ============================================================
# GET /info — возвращает метаданные о модели
# Полезно для проверки конфигурации

import json

r = httpx.get('http://localhost:8000/info')

print(f'Метод: GET /info')
print(f'Статус: {r.status_code}')
print(f'Ответ:')
print(json.dumps(r.json(), indent=2, ensure_ascii=False))
print()
print('Здесь мы видим:')
print('  - classes: список классов модели')
print('  - confidence_threshold: порог уверенности (70%)')
print('  - max_image_size_mb: максимальный размер файла')
print('  - allowed_types: допустимые форматы изображений')

In [ ]:
# ============================================================
# ТЕСТ 3: Классификация реальной картинки
# ============================================================
# POST /predict — отправляем картинку и получаем предсказание
# Это главный эндпоинт нашего сервиса

# Выбираем тестовую картинку
test_file = image_files[150]
true_label = '🐱 КОШКА' if is_cat(test_file.name) else '🐶 СОБАКА'

# Отправляем POST-запрос
# files={'file': (имя, содержимое, MIME-тип)} — формат multipart/form-data
with open(test_file, 'rb') as f:
    r = httpx.post('http://localhost:8000/predict',
                    files={'file': (test_file.name, f, 'image/jpeg')},
                    timeout=30)  # таймаут 30 сек (модель может думать долго)

result = r.json()

# Выводим результат
print(f'Метод: POST /predict')
print(f'Статус: {r.status_code}')
print(f'Файл: {test_file.name}')
print(f'Реальный класс: {true_label}')
print(f'Ответ сервера:')
print(json.dumps(result, indent=2, ensure_ascii=False))

# Визуализация: картинка + результат
img = PILImage.create(test_file)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# Картинка
ax[0].imshow(img)
ax[0].axis('off')
emoji = '🐱' if result.get('class') == 'кот' else '🐶'
ax[0].set_title(f'{emoji} {result.get("class")} ({result.get("confidence", 0):.1%})',
               fontsize=13, fontweight='bold')

# Вероятности
probs = result.get('all_probs', {})
if probs:
    ax[1].barh(list(probs.keys()), list(probs.values()), color=['#4ECDC4', '#FF6B6B'])
    ax[1].set_xlim(0, 1)
    ax[1].set_xlabel('Вероятность')
    ax[1].axvline(x=0.7, color='red', linestyle='--', label='Порог 70%')
    ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ТЕСТ 4: Серия картинок — визуальная сетка результатов
# ============================================================
# Отправляем 9 случайных картинок и показываем результаты
# Это помогает оценить качество модели «на глаз»

import random
random.seed(42)  # фиксируем seed для воспроизводимости
test_indices = random.sample(range(len(image_files)), 9)

fig, axes = plt.subplots(3, 3, figsize=(14, 12))

correct = 0
total = 0

for idx, ax in zip(test_indices, axes.flat):
    f = image_files[idx]
    true_lbl = 'кот' if is_cat(f.name) else 'собака'
    
    # Отправляем запрос к API
    with open(f, 'rb') as fh:
        r = httpx.post('http://localhost:8000/predict',
                        files={'file': (f.name, fh, 'image/jpeg')}, timeout=30)
    result = r.json()
    
    # Показываем картинку
    img = PILImage.create(f)
    ax.imshow(img)
    ax.axis('off')
    
    # Формируем подпись
    pred_class = result.get('class', '?')
    confidence = result.get('confidence', 0)
    is_reliable = result.get('is_reliable', False)
    
    # Считаем точность
    if pred_class == true_lbl:
        correct += 1
        color = 'green'
    elif pred_class == 'не уверен':
        color = 'orange'
    else:
        color = 'red'
    total += 1
    
    emoji = '🐱' if pred_class == 'кот' else ('🐶' if pred_class == 'собака' else '❓')
    ax.set_title(f'{emoji} {pred_class} ({confidence:.0%})',
                color=color, fontsize=11, fontweight='bold')

plt.suptitle(f'Результаты API: {correct}/{total} правильных', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ТЕСТ 5: «Плохой» запрос — отправляем текст вместо картинки
# ============================================================
# В продакшене пользователи могут отправить что угодно!
# Наш сервис должен обработать это и вернуть понятную ошибку
# Не упасть, а вернуть HTTP 400 (Bad Request)

r = httpx.post('http://localhost:8000/predict',
               files={'file': ('test.txt', b'Just some text, not an image', 'text/plain')},
               timeout=10)

print(f'Отправили: текстовый файл (content_type: text/plain)')
print(f'Статус: {r.status_code}')
print(f'Ответ: {json.dumps(r.json(), indent=2, ensure_ascii=False)}')
print()
if r.status_code == 400:
    print('✅ Сервис вернул 400 Bad Request — правильно!')
    print('   Файл не является изображением, сервис вежливо отказал.')
else:
    print('⚠️ Ожидали статус 400, получили', r.status_code)

In [ ]:
# ============================================================
# ТЕСТ 6: Слишком большой файл
# ============================================================
# Ограничение размера файла — защита от DDoS и нехватки памяти
# Наш лимит: 10 МБ (MAX_IMAGE_SIZE в model_service.py)
# Отправляем файл 11 МБ — сервис должен отказать

big_data = b'x' * (11 * 1024 * 1024)  # 11 МБ мусорных данных

r = httpx.post('http://localhost:8000/predict',
               files={'file': ('big.jpg', big_data, 'image/jpeg')},
               timeout=10)

print(f'Отправили: файл 11 МБ (лимит: 10 МБ)')
print(f'Статус: {r.status_code}')
print(f'Ответ: {json.dumps(r.json(), indent=2, ensure_ascii=False)}')
print()
if r.status_code == 400:
    print('✅ Защита от больших файлов работает!')
    print('   Сервис вернул 400 с объяснением причины.')
else:
    print('⚠️ Ожидали статус 400, получили', r.status_code)

In [ ]:
# ============================================================
# Останавливаем сервер
# ============================================================
# После тестов сервер нужно остановить — он занимает порт 8000
# pkill находит процесс по шаблону имени и отправляет сигнал завершения

!pkill -f 'uvicorn.*main:app' 2>/dev/null || true
# 2>/dev/null — подавляем вывод ошибок (если процесса нет)
# || true — команда не падает, если pkill не нашёл процесс

print('✅ Сервер остановлен.')
print('   Если нужно перезапустить — выполните ячейку запуска выше.')

---
## Шаг 12. Типичные ошибки и как их читать

В реальной разработке вы будете видеть ошибки чаще, чем работающий код. Это нормально! Главное — уметь их читать и понимать, что сломалось.

### Правила чтения ошибок

1. **Смотрите на ПОСЛЕДНЮЮ строку** — там тип ошибки и сообщение
2. **Смотрите на СТЕК-ТРЕЙС** (снизу вверх) — откуда пришла ошибка
3. **Ищите ВАШИ файлы** в стек-трейсе — ошибка в вашем коде или в библиотеке?
4. **Гуглите текст ошибки** — скорее всего, кто-то уже решал эту проблему


In [ ]:
# ============================================================
# ИНТЕРАКТИВНЫЙ: Справочник типичных ошибок
# ============================================================
# Раскройте каждый пункт, чтобы увидеть:
# - Как выглядит ошибка
# - Что она означает
# - Как её исправить

errors_accordion = widgets.Accordion(children=[
    widgets.HTML('''
        <h4>❌ FileNotFoundError: No such file or directory: 'cat_dog_model.pkl'</h4>
        <p><b>Когда:</b> Файл модели удалён, перемещён или не был скопирован на сервер.</p>
        <p><b>Как читать:</b></p>
        <ol>
          <li><code>FileNotFoundError</code> — файл не найден</li>
          <li>Путь в сообщении — именно этот файл ищется</li>
          <li><b>Решение:</b> проверить, что модель лежит по нужному пути</li>
        </ol>
        <p><b>Частая причина:</b> модель обучена на одной машине, а сервис запускается на другой.
        Не забыли скопировать .pkl файл?</p>
    '''),
    widgets.HTML('''
        <h4>❌ PIL.UnidentifiedImageError: cannot identify image file</h4>
        <p><b>Когда:</b> Пользователь загрузил файл, который НЕ является картинкой
        (PDF, текст, повреждённый JPEG).</p>
        <p><b>Как читать:</b></p>
        <ol>
          <li><code>UnidentifiedImageError</code> — Pillow не смог распознать формат</li>
          <li>Файл существует, но его содержимое — не изображение</li>
          <li><b>Решение:</b> валидация content_type + try/except при открытии</li>
        </ol>
        <p><b>Как мы решили:</b> в <code>predict()</code> стоит try/except,
        который возвращает понятную ошибку вместо падения.</p>
    '''),
    widgets.HTML('''
        <h4>❌ RuntimeError: CUDA out of memory</h4>
        <p><b>Когда:</b> Не хватает видеопамяти (GPU RAM) для обработки запроса.</p>
        <p><b>Как читать:</b></p>
        <ol>
          <li><code>CUDA</code> — это GPU, <code>out of memory</code> — не хватает памяти</li>
          <li>Модель + входные данные не помещаются в GPU</li>
          <li><b>Решение:</b> уменьшить batch_size, использовать CPU, или добавить GPU</li>
        </ol>
        <p><b>Совет:</b> в продакшене часто ограничивают размер входного изображения,
        чтобы избежать этой ошибки.</p>
    '''),
    widgets.HTML('''
        <h4>❌ HTTP 422 Unprocessable Entity</h4>
        <p><b>Когда:</b> FastAPI не смог валидировать входные данные.</p>
        <p><b>Как читать:</b></p>
        <ol>
          <li><code>422</code> — данные есть, но неправильного формата</li>
          <li>В ответе будет поле <code>detail</code> с описанием проблемы</li>
          <li><b>Решение:</b> проверить, что отправляете данные в правильном формате</li>
        </ol>
        <p><b>Пример:</b> забыли добавить <code>file=</code> в multipart-запрос.</p>
    '''),
    widgets.HTML('''
        <h4>❌ ConnectionError: Connection refused</h4>
        <p><b>Когда:</b> Сервер не запущен, или запущен на другом порту.</p>
        <p><b>Как читать:</b></p>
        <ol>
          <li><code>Connection refused</code> — никто не слушает на этом порту</li>
          <li>Проверьте: запущен ли <code>uvicorn</code>?</li>
          <li><b>Решение:</b> запустить сервер или проверить порт</li>
        </ol>
        <p><b>Совет:</b> в Colab сервер может «заснуть» после периода неактивности.</p>
    '''),
])

error_titles = [
    '❌ FileNotFoundError',
    '❌ PIL.UnidentifiedImageError',
    '❌ CUDA out of memory',
    '❌ HTTP 422',
    '❌ Connection refused',
]
for i, title in enumerate(error_titles):
    errors_accordion.set_title(i, title)

display(errors_accordion)

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ: Шпаргалка HTTP-кодов
# ============================================================
# HTTP-коды — это «язык», на котором сервер говорит клиенту
# о результате запроса. Знать их — обязательно!

fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

codes = [
    ('200', 'OK',             'Всё хорошо, вот результат',             '#90EE90'),
    ('400', 'Bad Request',    'Клиент отправил ерунду (плохой файл)',  '#FFFF90'),
    ('404', 'Not Found',      'Такого эндпоинта нет (опечатка в URL)', '#FFFF90'),
    ('422', 'Unprocessable',  'Данные есть, но неправильного формата', '#FFD700'),
    ('500', 'Server Error',   'Баг на стороне сервера (модель упала)', '#FFB3BA'),
]

for i, (code, name, desc, color) in enumerate(codes):
    y = 5.0 - i * 1.0
    
    # Код
    ax.add_patch(plt.Rectangle((0.5, y-0.35), 1.5, 0.7, fc=color, ec='gray', lw=2, alpha=0.8))
    ax.text(1.25, y, code, ha='center', va='center', fontsize=16, fontweight='bold')
    
    # Название
    ax.text(2.5, y, name, va='center', fontsize=13, fontweight='bold', color='steelblue')
    
    # Описание
    ax.text(6.0, y, desc, va='center', fontsize=11, color='#333')
    
    # Линия-разделитель
    if i < len(codes) - 1:
        ax.axhline(y=y-0.5, xmin=0.05, xmax=0.95, color='lightgray', lw=0.5)

ax.set_title('HTTP-коды: шпаргалка', fontsize=16, fontweight='bold', pad=15)
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
plt.tight_layout()
plt.show()

print('Запомните: 2xx = хорошо, 4xx = ошибка клиента, 5xx = ошибка сервера')

---
## Шаг 13. Практические задания

Теперь ваша очередь! Каждое задание — это возможность **сломать код, починить и научиться**.

> Принцип: вы — автор. Меняйте код, наблюдайте результат, делайте выводы.


In [ ]:
# ============================================================
# ИНТЕРАКТИВНЫЙ: Список заданий с подсказками
# ============================================================

tasks = widgets.Accordion(children=[
    widgets.HTML('''
        <h4>🎯 Задание 1: Поиск «золотого» порога</h4>
        <p>В <code>model_service.py</code> стоит <code>CONFIDENCE_THRESHOLD = 0.7</code>. 
        Попробуйте:</p>
        <ul>
          <li><b>0.3</b> — почти все предсказания «надёжные». Сколько ошибок?</li>
          <li><b>0.95</b> — модель часто не уверена. Сколько отказов?</li>
          <li>Найдите «золотую середину»</li>
        </ul>
        <p><b>Подсказка:</b> протестируйте на 50 картинках и посчитайте: 
        (а) сколько «не уверен», (б) сколько ошибок среди уверенных.</p>
        <p><b>Код для исследования:</b></p>
        <pre>
# Измените порог в model_service.py:
# CONFIDENCE_THRESHOLD = 0.3  # потом 0.5, 0.7, 0.95

# Протестируйте:
results = []
for i in range(50):
    with open(image_files[i], 'rb') as f:
        r = httpx.post('http://localhost:8000/predict',
                       files={'file': (image_files[i].name, f, 'image/jpeg')})
        results.append(r.json())

# Посчитайте:
unsure = sum(1 for r in results if r.get('class') == 'не уверен')
errors = sum(1 for r in results if r.get('class') != 'не уверен' 
             and r.get('class') != ('кот' if is_cat(image_files[i].name) else 'собака'))
print(f'Не уверен: {unsure}/50, Ошибки: {errors}/{50-unsure}')
        </pre>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 2: Добавьте эндпоинт /predict_batch</h4>
        <p>Сейчас API принимает одну картинку. Добавьте эндпоинт для 
        <b>нескольких картинок сразу</b>:</p>
        <ul>
          <li>Принимает список файлов (до 10 штук)</li>
          <li>Возвращает список предсказаний</li>
          <li>Добавьте ограничение: не больше 10 файлов</li>
        </ul>
        <p><b>Подсказка:</b></p>
        <pre>
@app.post("/predict_batch")
async def predict_batch(files: list[UploadFile] = File(...)):
    if len(files) > 10:
        raise HTTPException(400, "Максимум 10 файлов")
    results = []
    for file in files:
        image_bytes = await file.read()
        result = predict(image_bytes)
        results.append(result)
    return {"predictions": results}
        </pre>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 3: Логирование запросов</h4>
        <p>Добавьте логирование каждого запроса:</p>
        <ul>
          <li>Время запроса</li>
          <li>Имя файла и размер</li>
          <li>Результат предсказания</li>
          <li>Время обработки</li>
        </ul>
        <p><b>Подсказка:</b> используйте middleware FastAPI:</p>
        <pre>
import time
import logging

@app.middleware("http")
async def log_requests(request, call_next):
    start = time.time()
    response = await call_next(request)
    duration = time.time() - start
    logging.info(f"{request.method} {request.url} "
                 f"-> {response.status_code} ({duration:.3f}s)")
    return response
        </pre>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 4: Обработка «мусорных» картинок</h4>
        <p>Модель ВСЕГДА даёт ответ, даже если подать шум. 
        Добавьте проверку:</p>
        <ul>
          <li>Если уверенность < 0.5 — вернуть «не изображение животного»</li>
          <li>Добавьте тест: подайте случайный шум и проверьте ответ</li>
          <li>Подумайте: что ещё можно проверять?</li>
        </ul>
        <p><b>Идеи для расширения:</b></p>
        <ul>
          <li>Проверка соотношения сторон (кошки не бывают 1x100 пикселей)</li>
          <li>Проверка «разнообразия» пикселей (однотонная картинка — не фото)</li>
          <li>Запуск детектора объектов перед классификатором</li>
        </ul>
    '''),
    widgets.HTML('''
        <h4>🎯 Задание 5: Деплой на Hugging Face Spaces</h4>
        <p>Задеплойте сервис на <b>Hugging Face Spaces</b> — 
        бесплатный хостинг для ML-моделей!</p>
        <ol>
          <li>Создайте аккаунт на <a href="https://huggingface.co">huggingface.co</a></li>
          <li>Создайте Space (Docker SDK)</li>
          <li>Загрузите файлы: model_service.py, main.py, requirements.txt, модель</li>
          <li>Добавьте Dockerfile</li>
          <li>Протестируйте по публичному URL</li>
        </ol>
        <p><b>Пример Dockerfile:</b></p>
        <pre>
FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]
        </pre>
    '''),
])

task_titles = [
    '🎯 Задание 1: Поиск «золотого» порога',
    '🎯 Задание 2: Эндпоинт /predict_batch',
    '🎯 Задание 3: Логирование запросов',
    '🎯 Задание 4: Обработка «мусорных» картинок',
    '🎯 Задание 5: Деплой на Hugging Face Spaces',
]
for i, title in enumerate(task_titles):
    tasks.set_title(i, title)

display(tasks)

---
## Что дальше?

<img src="https://sfile.chatglm.cn/images-ppt/2d6b62db5b30.png" width="500" align="center">

Вы научились:
- Обучать модель классификации изображений с fastai
- Сохранять и загружать модель
- Различать блокнот и продакшен
- Писать FastAPI-сервис с валидацией и обработкой ошибок
- Тестировать API (включая «плохие» запросы)

### Идеи для дальнейшего изучения

| Направление | Что изучить | Ресурс |
|------------|------------|--------|
| **Углублённый PyTorch** | Свёрточные сети «с нуля» | Следующий блокнот курса |
| **Docker** | Контейнеризация ML-сервисов | Docker Docs |
| **MLOps** | CI/CD для моделей, мониторинг | MLflow, Weights & Biases |
| **Продвинутый FastAPI** | Аутентификация, вебсокеты, фоновые задачи | FastAPI Docs |
| **Оптимизация модели** | ONNX, TensorRT, квантизация | ONNX Runtime |

> Следующий блокнот: **Свёрточные нейронные сети (CNN)** — разберём, как именно модель «видит» картинки.
